In [2]:
%pip install tensorflow scikit-learn matplotlib

INFO: pip is looking at multiple versions of ml-dtypes to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 MB 50.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 60.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 64.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 31.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.3
    Uninstalling protobuf-4.25.3:
      Successfully uninstalled protobuf-4.25.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 26.3 which is incompatible.

In [3]:
# Import the required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import mean_absolute_error, mean_squared_error


: 

In [3]:
# Load the requested data sets.
trns_fraud_df = pd.read_csv('/Users/raj/Desktop/CapStone/bankOps-platform/DataSets/transactions_fraud_train.csv')
infra_metrics_df = pd.read_csv('/Users/raj/Desktop/CapStone/bankOps-platform/DataSets/infrastructure_metrics.csv')
app_logs_df = pd.read_csv('/Users/raj/Desktop/CapStone/bankOps-platform/DataSets/application_logs.csv')

In [5]:
infra_metrics_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10080 entries, 0 to 10079
Data columns (total 13 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   timestamp                           10080 non-null  object 
 1   cpu_utilization                     10080 non-null  float64
 2   memory_usage                        10080 non-null  float64
 3   db_connection_pool_usage            10080 non-null  float64
 4   network_throughput_mbps             10080 non-null  float64
 5   application_error_rate              10080 non-null  int64  
 6   neft_success_rate                   10080 non-null  float64
 7   imps_success_rate                   10080 non-null  float64
 8   rtgs_success_rate                   10080 non-null  float64
 9   avg_transaction_processing_time_ms  10080 non-null  float64
 10  auth_success_rate                   10080 non-null  float64
 11  transaction_volume_per_hour         10080

Since timestamp is present as object, lets first convert the timestamp to Pandas Time Stamp.

In [6]:
infra_metrics_df["timestamp"] = pd.to_datetime(
    infra_metrics_df["timestamp"]
)

infra_metrics_df = (
    infra_metrics_df
    .sort_values("timestamp")
    .set_index("timestamp")
)

We are asked to perform Time series forecasting for transaction success rate, API Latency and Resource Utilization. So columns that we need to target would be cpu_utilization, memory_usage, db_connection_pool_usage, neft_success_rate, imps_success_rate, rtgs_success_rate, avg_transaction_processing_time_ms

In [7]:
target_columns = [
    "neft_success_rate",
    "imps_success_rate",
    "rtgs_success_rate",
    "avg_transaction_processing_time_ms",
    "cpu_utilization",
    "memory_usage",
    "db_connection_pool_usage"
]

target_df = infra_metrics_df[target_columns].copy()

There are no empty values for this data set, so we should be good and we should able to proceed with the next steps. 

In [11]:
lookback = 60

split_index = int(len(target_df)*0.8)

train_data = target_df.iloc[:split_index]

In [13]:
scaler = MinMaxScaler(feature_range=(0, 1))

scaler.fit(train_data)

scaled_data = scaler.transform(target_df)

In [14]:
X = []
y = []
target_indices = []

for i in range(lookback, len(scaled_data)):

    X.append(scaled_data[i-lookback:i])

    y.append(scaled_data[i])

    target_indices.append(i)

X = np.array(X)
y = np.array(y)
target_indices = np.array(target_indices)

In [15]:
train_mask = target_indices < split_index
test_mask = target_indices >= split_index

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (8004, 60, 7)
X_test: (2016, 60, 7)


In [2]:
model = Sequential()

model.add(
    LSTM(
        64,
        return_sequences=True,
        input_shape=(lookback, len(target_columns))
    )
)

model.add(Dropout(0.2))

model.add(
    LSTM(32, return_sequences=False)
)

model.add(Dropout(0.2))

model.add(Dense(16, activation="relu"))

model.add(Dense(len(target_columns)))

model.compile(
    optimizer="adam",
    loss="mean_squared_error",
    metrics=["mae"]
)

model.summary()

NameError: name 'Sequential' is not defined

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
predictions = model.predict(X_test)

# Convert predictions back to original units
predictions_actual = scaler.inverse_transform(predictions)

# Convert actual values back to original units
y_test_actual = scaler.inverse_transform(y_test)

In [ ]:
for i, column in enumerate(target_columns):

    mae = mean_absolute_error(
        y_test_actual[:, i],
        predictions_actual[:, i]
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test_actual[:, i],
            predictions_actual[:, i]
        )
    )

    print(f"\n{column}")
    print("MAE:", round(mae, 3))
    print("RMSE:", round(rmse, 3))

In [ ]:
column = "cpu_utilization"

column_index = target_columns.index(column)

plt.figure(figsize=(14, 5))

plt.plot(
    target_df.index[target_indices[test_mask]],
    y_test_actual[:, column_index],
    label="Actual"
)

plt.plot(
    target_df.index[target_indices[test_mask]],
    predictions_actual[:, column_index],
    label="Predicted"
)

plt.title("LSTM - Actual vs Predicted CPU Utilization")
plt.xlabel("Timestamp")
plt.ylabel("CPU Utilization")
plt.legend()
plt.show()

In [ ]:
# Create model artifact
timeseires_artifact = {
    'model': model
}

# Save model artifact
joblib.dump(timeseires_artifact, 'timeseries_forecasting.pkl')

print("Time Series Forecast Model Saved successfully.")
